# Run the factorial experiment

36 runs: 4 architectures x 3 split arms x 3 seeds.

Resumable. Results append to `results_nlst.csv` on **Drive**, so a Colab
disconnect costs at most the run in progress. Re-run cell 3 to continue.

Set the runtime to **GPU** first: Runtime -> Change runtime type -> T4 GPU.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/elhamum/lungcnet-leakage.git
%cd /content/lungcnet-leakage
!pip install -q pyyaml

import tensorflow as tf, os
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE -- enable the T4!")

os.makedirs("/content/drive/MyDrive/leakage_bench", exist_ok=True)
RESULTS = "/content/drive/MyDrive/leakage_bench/results_nlst.csv"
print("results ->", RESULTS)

## 2. Freeze the splits

Regenerates the same nine arm x seed columns from the committed manifest.
Deterministic, so it reproduces the splits already in the repo.

In [ ]:
!python src/make_splits.py --config configs/nlst.yaml

## 3. Run the queue

Arms sweep innermost, so an interrupted queue still leaves complete arm
comparisons for the architectures it finished.

Add `--limit 6` to cap how many runs happen in one session.

In [ ]:
!PYTHONPATH=src python src/runner.py --config configs/nlst.yaml --results "/content/drive/MyDrive/leakage_bench/results_nlst.csv"

## 4. Where things stand

In [ ]:
import pandas as pd
r = pd.read_csv(RESULTS)
print(f"{len(r)} runs recorded, {(r.status=='ok').sum()} ok\n")

ok = r[r.status == "ok"]
if len(ok):
    print("macro F1 by architecture and split arm:")
    print(ok.pivot_table(index="arch", columns="arm", values="test_macro_f1",
                         aggfunc=["mean","std","count"]).round(2).to_string())

    m = ok.groupby(["arch","arm"]).test_macro_f1.mean().unstack()
    if {"slice","patient"}.issubset(m.columns):
        print("\ninflation from leakage (macro F1 points):")
        cols = {"slice - patient": (m["slice"] - m["patient"]).round(2)}
        if "exam" in m.columns:
            cols["exam - patient"] = (m["exam"] - m["patient"]).round(2)
        print(pd.DataFrame(cols).to_string())

    if ok.collapsed_to_single_class.any():
        print("\ncollapsed runs (report these, do not silently average them):")
        print(ok[ok.collapsed_to_single_class]
              [["arch","arm","seed","val_accuracy"]].to_string(index=False))